# 💓 BioEcho v2 — Notebook 2: rPPG / Vital Signs Model
**Detects:** Heart Rate · HRV (SDNN) · SpO2 · Blood Pressure (systolic)

## Datasets (all free, no email required)
| Dataset | Size | Source | Registration? |
|---|---|---|---|
| **UBFC-rPPG** | ~3.5 GB | Kaggle `toazismail/ubfc-rppg` | Kaggle account only |
| **MMPD** | ~8 GB | Kaggle `jacktangthu/mmpd-rppg` | Kaggle account only |
| **MCD-rPPG** | ~50 GB | HuggingFace `kyegorov/mcd_rppg` | HF account only |
| **SCAMPS** | ~25 GB | GitHub link (see cell below) | None |
| **Synthetic** | generated | Built-in | None |

## 2× T4 GPU Optimisations
- `torch.compile` → `DataParallel` (correct order)
- `batch_size=8`, `grad_accum=2`, `num_workers=4`
- `persistent_workers`, `prefetch_factor=2`, `non_blocking=True`
- CUDA 12.x wheel (`cu121`), BF16, gradient checkpointing

In [ ]:
import os, json, subprocess
from pathlib import Path

# ── Kaggle credentials
kd = Path.home() / '.kaggle'
kd.mkdir(exist_ok=True)
cp = kd / 'kaggle.json'
# On Kaggle, credentials are auto-injected. No hardcoded key needed.
import os
json.dump({'username': os.environ.get('KAGGLE_USERNAME', 'saroopmandal'),
           'key': os.environ.get('KAGGLE_KEY', '')}, open(cp, 'w'))
cp.chmod(0o600)

# ── HuggingFace token (for MCD-rPPG — free account)
# If you don't have one yet: https://huggingface.co/settings/tokens
HF_TOKEN = ''   # ← paste your HF read token here (leave blank to skip MCD-rPPG)

# ── SCAMPS direct URL
# Get the current link from: https://github.com/danmcduff/scampsdataset
# (Microsoft Azure Blob — link is in the README, no login needed)
SCAMPS_URL = ''  # ← paste the .tar.gz URL from the GitHub README (leave blank to skip)

print('✅ Config set')

In [ ]:
def run(cmd, timeout=900):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
    if r.returncode != 0 and r.stderr:
        print(f'WARN [{cmd[:60]}]: {r.stderr[:300]}')
    return r

# ── cu121 wheel: Kaggle T4 runs CUDA 12.x
run('pip install -q --upgrade pip')
run('pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121', timeout=600)
run('pip install -q bitsandbytes>=0.43.0')
run('pip install -q opencv-python-headless mediapipe')
run('pip install -q scipy scikit-learn einops rich matplotlib seaborn pandas torchmetrics')
run('pip install -q onnx onnxruntime-gpu')
run('pip install -q huggingface_hub datasets')   # for MCD-rPPG
run('pip install -q gdown')                       # for any GDrive fallbacks

from onnxruntime.quantization import quantize_dynamic, QuantType
print('✅ All packages installed')

In [ ]:
import gc, time, warnings, random, math, subprocess, tarfile, zipfile
from copy import deepcopy
from dataclasses import dataclass
from typing import Dict, List, Optional

import numpy as np
import pandas as pd
import cv2
import scipy.io
import matplotlib.pyplot as plt
from scipy import signal as scipy_signal
from scipy.signal import butter, filtfilt
from rich.console import Console
from rich.table import Table

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import bitsandbytes as bnb
import onnx
import onnxruntime as ort

warnings.filterwarnings('ignore')
console = Console()

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

torch.backends.cudnn.benchmark     = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32    = True

NUM_GPUS = torch.cuda.device_count()
DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DTYPE    = torch.float16

console.print(f'[bold green]GPUs: {NUM_GPUS} | BF16 | Device: {DEVICE}[/]')
for i in range(NUM_GPUS):
    p = torch.cuda.get_device_properties(i)
    console.print(f'  GPU {i}: {p.name} | {p.total_memory/1e9:.1f} GB')

In [ ]:
@dataclass
class rPPGConfig:
    data_dir:  str = '/kaggle/working/bioecho/rppg/data'
    ckpt_dir:  str = '/kaggle/working/bioecho/rppg/checkpoints'
    out_dir:   str = '/kaggle/working/bioecho/rppg/outputs'

    fps:       int   = 30
    clip_len:  int   = 300          # 10 s @ 30 fps
    face_H:    int   = 128
    face_W:    int   = 128

    bpf_low:   float = 0.7
    bpf_high:  float = 3.5
    bpf_order: int   = 4

    embed_dim:    int   = 256
    se_reduction: int   = 16
    dropout:      float = 0.3

    # ── 2× T4 settings
    batch_size:  int   = 16          # DataParallel → 4 per GPU
    grad_accum:  int   = 1          # effective batch = 16
    num_workers: int   = 4          # 2 per GPU

    epochs:       int   = 15
    lr:           float = 1e-4
    weight_decay: float = 1e-4
    grad_clip:    float = 1.0
    patience:     int   = 10
    ema_decay:    float = 0.9995

C = rPPGConfig()
for d in [C.data_dir, C.ckpt_dir, C.out_dir]:
    Path(d).mkdir(parents=True, exist_ok=True)

console.print(f'[bold]Effective batch: {C.batch_size * C.grad_accum} | Per-GPU: {C.batch_size // max(NUM_GPUS,1)}[/]')

In [ ]:
CKPT_DIR = Path(C.ckpt_dir)

def save_checkpoint(epoch, model_state, ema_shadow, opt_state, sch_state, scaler_state, history, best_val):
    ckpt = {'epoch': epoch, 'model_state': model_state, 'ema_shadow': ema_shadow,
            'opt_state': opt_state, 'sch_state': sch_state, 'scaler_state': scaler_state,
            'history': history, 'best_val': best_val}
    p = CKPT_DIR / f'rppg_epoch_{epoch:03d}.pt'
    torch.save(ckpt, p)
    old = sorted(CKPT_DIR.glob('rppg_epoch_*.pt'),
                 key=lambda x: int(x.stem.split('_')[-1]))[:-2]
    for o in old: o.unlink(missing_ok=True)
    return p

def find_latest(prefix='rppg_epoch'):
    ckpts = sorted(CKPT_DIR.glob(f'{prefix}_*.pt'),
                   key=lambda x: int(x.stem.split('_')[-1]))
    return ckpts[-1] if ckpts else None

latest = find_latest()
RESUME_CKPT = str(latest) if latest else ''
console.print(f'[{"yellow" if latest else "green"}]{"▶ Resume: " + latest.name if latest else "✅ Fresh run"}[/]')

In [ ]:
import urllib.request
DATA = Path(C.data_dir)

# ─────────────────────────────────────────────────────────────
# Helper
# ─────────────────────────────────────────────────────────────
def kaggle_dl(slug, dest, desc=''):
    """Download a Kaggle dataset by slug (user/dataset-name)."""
    dest = Path(dest)
    if dest.exists() and any(dest.iterdir()):
        console.print(f'[yellow]Skip (exists): {desc}[/]'); return True
    dest.mkdir(parents=True, exist_ok=True)
    console.print(f'[cyan]Downloading {desc} from Kaggle...[/]')
    r = subprocess.run(
        f'kaggle datasets download -d {slug} -p {dest} --unzip',
        shell=True, capture_output=True, text=True, timeout=1800
    )
    ok = r.returncode == 0
    if ok: console.print(f'[green]✅ {desc} downloaded[/]')
    else:  console.print(f'[red]❌ {desc}: {r.stderr[:200]}[/]')
    return ok


# ─────────────────────────────────────────────────────────────
# 1. UBFC-rPPG  (Kaggle, ~3.5 GB)
#    No email — Kaggle account only
# ─────────────────────────────────────────────────────────────
# Try Kaggle input first, then download
ubfc_input = Path('/kaggle/input/ubfc-rppg')
if ubfc_input.exists():
    ubfc_ok = True
    console.print('[green]✅ UBFC-rPPG found in Kaggle input[/]')
else:
    ubfc_ok = kaggle_dl('toazismail/ubfc-rppg', DATA / 'ubfc', 'UBFC-rPPG')


# ─────────────────────────────────────────────────────────────
# 2. MMPD  (Kaggle, ~8 GB)
#    No email — Kaggle account only
#    660 videos · Fitzpatrick skin 3-6 · 4 lighting · 4 activities
# ─────────────────────────────────────────────────────────────
mmpd_ok = kaggle_dl('jacktangthu/mmpd-rppg', DATA / 'mmpd', 'MMPD')


# ─────────────────────────────────────────────────────────────
# 3. SCAMPS  (direct Azure Blob, ~25 GB) — NO registration
#    Get URL from: https://github.com/danmcduff/scampsdataset
#    Set SCAMPS_URL in the first cell, then this cell does the rest.
# ─────────────────────────────────────────────────────────────
scamps_ok = False
scamps_dir = DATA / 'scamps'
if SCAMPS_URL:
    tar_path = DATA / 'scamps_videos.tar.gz'
    if not tar_path.exists():
        console.print('[cyan]Downloading SCAMPS (~25 GB)...[/]')
        r = subprocess.run(f'wget -q -O {tar_path} "{SCAMPS_URL}"',
                           shell=True, timeout=7200)
        if r.returncode != 0:
            console.print('[red]❌ SCAMPS download failed — check SCAMPS_URL[/]')
    if tar_path.exists() and tar_path.stat().st_size > 1e6:
        scamps_dir.mkdir(exist_ok=True)
        console.print('[cyan]Extracting SCAMPS...[/]')
        subprocess.run(f'tar -xzf {tar_path} -C {scamps_dir}',
                       shell=True, timeout=3600)
        scamps_ok = any(scamps_dir.rglob('*.mat'))
        if scamps_ok: console.print('[green]✅ SCAMPS extracted[/]')
else:
    console.print('[yellow]SCAMPS skipped — set SCAMPS_URL in cell 1[/]')


# ─────────────────────────────────────────────────────────────
# 4. MCD-rPPG  (HuggingFace Hub)
#    Free HF account — 600 subjects, 13 biomarkers incl. BP + SpO2
#    Set HF_TOKEN in cell 1
# ─────────────────────────────────────────────────────────────
mcd_ok = False
mcd_dir = DATA / 'mcd_rppg'
if HF_TOKEN:
    try:
        from huggingface_hub import snapshot_download
        console.print('[cyan]Downloading MCD-rPPG from HuggingFace...[/]')
        snap = snapshot_download(
            repo_id='kyegorov/mcd_rppg',
            repo_type='dataset',
            local_dir=str(mcd_dir),
            token=HF_TOKEN,
            ignore_patterns=['*.zip'],  # skip raw zip if videos already there
        )
        mcd_ok = True
        console.print(f'[green]✅ MCD-rPPG downloaded → {snap}[/]')
    except Exception as e:
        console.print(f'[red]❌ MCD-rPPG: {e}[/]')
else:
    console.print('[yellow]MCD-rPPG skipped — set HF_TOKEN in cell 1[/]')


# ─────────────────────────────────────────────────────────────
# 5. Synthetic fallback (always generated)
# ─────────────────────────────────────────────────────────────
synth_dir = DATA / 'synthetic'
synth_dir.mkdir(exist_ok=True)

def gen_synth_clip(T=300, fps=30, hr=None, bp_sys=120.0, hrv_sdnn=45.0):
    if hr is None: hr = np.random.uniform(50, 110)
    hr_hz = hr / 60.0
    t = np.arange(T) / fps
    bvp = (np.sin(2*np.pi*hr_hz*t) + 0.3*np.sin(4*np.pi*hr_hz*t)
           + 0.05*np.random.randn(T))
    bvp *= 1.0 + (hrv_sdnn/1000)*np.sin(2*np.pi*0.1*t)
    skin   = np.array([200, 160, 140], dtype=np.float32)
    frames = np.tile(skin, (T, C.face_H, C.face_W, 1)).copy()
    frames[:,:,:,1] += bvp[:,None,None] * 10.0
    frames[:,:,:,0] += bvp[:,None,None] * 2.0
    frames += np.random.randn(*frames.shape) * 3.0
    return np.clip(frames,0,255).astype(np.uint8), bvp.astype(np.float32), hr, bp_sys, hrv_sdnn

real_sources = ubfc_ok or mmpd_ok or scamps_ok or mcd_ok
N_SYNTH = 100 if real_sources else 500
synth_meta = []
for i in range(N_SYNTH):
    hr_ = np.random.uniform(50,110); bp_ = np.random.uniform(100,160); hrv_ = np.random.uniform(20,80)
    frames, bvp, hr_, bp_, hrv_ = gen_synth_clip(C.clip_len, C.fps, hr_, bp_, hrv_)
    np.save(synth_dir/f'clip_{i:04d}.npy', frames)
    np.save(synth_dir/f'bvp_{i:04d}.npy',  bvp)
    synth_meta.append({'id':f'{i:04d}','hr':hr_,'bp_systolic':bp_,'hrv_sdnn':hrv_})
    if i%100==0: console.print(f'  Synth {i}/{N_SYNTH}')
pd.DataFrame(synth_meta).to_csv(synth_dir/'meta.csv', index=False)
console.print(f'[green]✅ {N_SYNTH} synthetic clips[/]')

# ── Summary
t = Table(title='Dataset Status')
t.add_column('Dataset'); t.add_column('Status'); t.add_column('Notes')
t.add_row('UBFC-rPPG',  '✅' if ubfc_ok  else '⏭ skipped', 'Real face videos, 42 subjects')
t.add_row('MMPD',       '✅' if mmpd_ok  else '⏭ skipped', '660 videos, diverse skin tones')
t.add_row('SCAMPS',     '✅' if scamps_ok else '⏭ set SCAMPS_URL', '2800 synthetic, HRV labels')
t.add_row('MCD-rPPG',   '✅' if mcd_ok   else '⏭ set HF_TOKEN',   '600 subjects, BP+SpO2 GT')
t.add_row('Synthetic',  '✅', f'{N_SYNTH} clips, always available')
console.print(t)

In [ ]:
# ── Signal processing utils
def bvp_to_hr(bvp, fps=30):
    N = len(bvp)
    freqs = np.fft.rfftfreq(N, 1.0/fps)
    mag   = np.abs(np.fft.rfft(bvp * np.hanning(N)))
    mask  = (freqs >= C.bpf_low) & (freqs <= C.bpf_high)
    if not mask.any(): return 70.0
    return float(freqs[mask][mag[mask].argmax()] * 60)

def bvp_to_hrv_sdnn(bvp, fps=30):
    peaks, _ = scipy_signal.find_peaks(bvp, distance=int(fps*0.4), prominence=0.05)
    if len(peaks) < 3: return 0.0
    return float(np.diff(peaks).std() / fps * 1000)

def bandpass(sig, fps=30):
    b, a = butter(C.bpf_order,
                  [C.bpf_low/(fps/2), C.bpf_high/(fps/2)], btype='bandpass')
    return filtfilt(b, a, sig).astype(np.float32)

In [ ]:
# ─────────────────────────────────────────────────────────────
# Dataset parsers  (each returns a list of record dicts)
# ─────────────────────────────────────────────────────────────

def parse_mat_frames(mat, key='Xsub'):
    """
    Read face frames from a .mat file.
    Both SCAMPS and MMPD store frames as [C,H,W,T] uint8 in 'Xsub'.
    Returns numpy array [T,H,W,C] uint8.
    """
    try:
        m = scipy.io.loadmat(mat, variable_names=[key, 'd_ppg', 'gt_HR', 'bpm'])
    except Exception:
        return None, None, 70.0

    frames_raw = m.get(key)
    if frames_raw is None: return None, None, 70.0

    # shape [3,H,W,T] → [T,H,W,3]
    if frames_raw.ndim == 4:
        frames = np.transpose(frames_raw, (3,1,2,0)).copy()
    else:
        frames = frames_raw  # fallback

    # PPG signal
    ppg = m.get('d_ppg', m.get('bvp', None))
    if ppg is not None:
        ppg = ppg.flatten().astype(np.float32)
        ppg = scipy_signal.resample(ppg, frames.shape[0]).astype(np.float32)
    else:
        ppg = np.zeros(frames.shape[0], np.float32)

    # HR
    hr_gt = m.get('gt_HR', m.get('bpm', None))
    hr = float(hr_gt.flatten()[0]) if hr_gt is not None else bvp_to_hr(ppg)
    return frames, ppg, hr


def load_scamps(root: Path):
    """SCAMPS: .mat files with Xsub frames + d_ppg + HRV stats."""
    recs = []
    for mat_path in sorted(root.rglob('*.mat')):
        frames, ppg, hr = parse_mat_frames(mat_path, key='Xsub')
        if frames is None: continue
        # SCAMPS also has d_sdnn (HRV SDNN) — use it if present
        try:
            m = scipy.io.loadmat(mat_path, variable_names=['d_sdnn'])
            hrv = float(m['d_sdnn'].flatten().mean()) if 'd_sdnn' in m else bvp_to_hrv_sdnn(ppg)
        except Exception:
            hrv = bvp_to_hrv_sdnn(ppg)
        recs.append({'_frames': frames, 'bvp': ppg, 'hr': hr,
                     'bp_systolic': 120.0, 'hrv_sdnn': hrv})
    console.print(f'[green]SCAMPS: {len(recs)} clips[/]')
    return recs


def load_mmpd(root: Path):
    """
    MMPD: per-subject folders with p{N}_{task}.mat files.
    Each .mat has 'video' [T,H,W,3] and 'gt_HR' scalar.
    """
    recs = []
    for mat_path in sorted(root.rglob('*.mat')):
        try:
            m = scipy.io.loadmat(mat_path,
                variable_names=['video', 'gt_HR', 'gt_PPG', 'Label_SkinTone'])
        except Exception: continue

        vid = m.get('video')
        if vid is None: continue
        # MMPD stores [T,H,W,3] directly
        frames = vid.astype(np.uint8)

        ppg = m.get('gt_PPG', None)
        if ppg is not None:
            ppg = scipy_signal.resample(ppg.flatten().astype(np.float32),
                                        frames.shape[0]).astype(np.float32)
        else:
            ppg = np.zeros(frames.shape[0], np.float32)

        hr_gt = m.get('gt_HR', None)
        hr = float(hr_gt.flatten()[0]) if hr_gt is not None else bvp_to_hr(ppg)
        recs.append({'_frames': frames, 'bvp': ppg, 'hr': hr,
                     'bp_systolic': 120.0, 'hrv_sdnn': bvp_to_hrv_sdnn(ppg)})
    console.print(f'[green]MMPD: {len(recs)} clips[/]')
    return recs


def load_ubfc(root: Path):
    """UBFC-rPPG: subject folders with vid.avi + ground_truth.txt"""
    recs = []
    for gt_file in sorted(root.rglob('ground_truth.txt')):
        vid_file = gt_file.parent / 'vid.avi'
        if not vid_file.exists(): continue
        try:
            gt = np.loadtxt(gt_file)
        except Exception: continue
        # UBFC gt: [bvp, hr, timestamps]
        bvp = gt[:, 0].astype(np.float32) if gt.ndim > 1 else gt.astype(np.float32)
        bvp = scipy_signal.resample(bvp, C.clip_len).astype(np.float32)
        hr  = gt[0, 1] if gt.ndim > 1 and gt.shape[1] > 1 else bvp_to_hr(bvp)
        recs.append({'video': str(vid_file), 'bvp': bvp,
                     'hr': float(hr), 'bp_systolic': 120.0,
                     'hrv_sdnn': bvp_to_hrv_sdnn(bvp)})
    console.print(f'[green]UBFC-rPPG: {len(recs)} clips[/]')
    return recs


def load_mcd(root: Path):
    """
    MCD-rPPG: CSV metadata + video files.
    Has ground-truth bp_systolic and spo2 — best fit for this model.
    """
    recs = []
    meta_files = list(root.rglob('*.csv'))
    if not meta_files: return recs
    meta = pd.concat([pd.read_csv(f) for f in meta_files], ignore_index=True)

    for _, row in meta.iterrows():
        # Flexible column names
        vid_col = next((c for c in meta.columns if 'video' in c.lower() or 'path' in c.lower()), None)
        if vid_col is None: continue
        vid_path = root / str(row[vid_col])
        if not vid_path.exists(): continue

        hr_col  = next((c for c in meta.columns if 'hr' in c.lower()  or 'heart' in c.lower()), None)
        bp_col  = next((c for c in meta.columns if 'sbp' in c.lower() or 'systolic' in c.lower()), None)
        hrv_col = next((c for c in meta.columns if 'hrv' in c.lower() or 'sdnn' in c.lower()), None)

        recs.append({
            'video':       str(vid_path),
            'bvp':         np.zeros(C.clip_len, np.float32),
            'hr':          float(row[hr_col])   if hr_col  else 70.0,
            'bp_systolic': float(row[bp_col])   if bp_col  else 120.0,
            'hrv_sdnn':    float(row[hrv_col])  if hrv_col else 45.0,
        })
    console.print(f'[green]MCD-rPPG: {len(recs)} clips[/]')
    return recs


# ─────────────────────────────────────────────────────────────
# Build combined record list
# ─────────────────────────────────────────────────────────────
recs = []

# Synthetic (always)
meta_df = pd.read_csv(synth_dir / 'meta.csv')
for _, row in meta_df.iterrows():
    cid = row['id']
    recs.append({'npy': str(synth_dir/f'clip_{cid}.npy'),
                 'bvp': np.load(synth_dir/f'bvp_{cid}.npy'),
                 'hr': float(row['hr']), 'bp_systolic': float(row['bp_systolic']),
                 'hrv_sdnn': float(row['hrv_sdnn'])})

if ubfc_ok:   recs += load_ubfc(DATA / 'ubfc')
if mmpd_ok:   recs += load_mmpd(DATA / 'mmpd')
if scamps_ok: recs += load_scamps(DATA / 'scamps')
if mcd_ok:    recs += load_mcd(DATA / 'mcd_rppg')

random.shuffle(recs)
console.print(f'[bold]Total records: {len(recs)}[/]')

In [ ]:
class rPPGDataset(Dataset):
    def __init__(self, records, is_train=True):
        self.recs     = records
        self.is_train = is_train

    def __len__(self): return len(self.recs)

    def __getitem__(self, i):
        rec  = self.recs[i]
        clip = self._load(rec)                                   # (T,3,H,W) float [0,1]
        bvp  = torch.from_numpy(rec.get('bvp', np.zeros(C.clip_len, np.float32)))
        hr   = torch.tensor(rec.get('hr',           70.0),  dtype=torch.float32)
        bp   = torch.tensor(rec.get('bp_systolic', 120.0) / 200., dtype=torch.float32)
        hrv  = torch.tensor(rec.get('hrv_sdnn',    45.0)  / 100., dtype=torch.float32)

        if self.is_train:
            # Temporal jitter
            if random.random() < 0.3 and clip.shape[0] > C.clip_len:
                st = random.randint(0, clip.shape[0] - C.clip_len)
                clip = clip[st:st + C.clip_len]
            # Brightness jitter
            if random.random() < 0.4:
                clip = (clip * random.uniform(0.8, 1.2)).clamp(0, 1)
            # Temporal CutMix
            if random.random() < 0.25:
                cut = random.randint(10, 60)
                st  = random.randint(0, C.clip_len - cut)
                clip[st:st + cut] = 0.0

        return {'clip': clip, 'bvp': bvp, 'hr': hr, 'bp_systolic': bp, 'hrv_sdnn': hrv}

    def _load(self, rec):
        # Pre-loaded numpy frames from .mat (SCAMPS / MMPD)
        if '_frames' in rec:
            arr = rec['_frames'][:C.clip_len]
        # Pre-saved .npy (synthetic)
        elif 'npy' in rec:
            arr = np.load(rec['npy'])[:C.clip_len]
        # Video file (UBFC / MCD-rPPG)
        elif 'video' in rec:
            cap = cv2.VideoCapture(rec['video'])
            fs  = []
            while len(fs) < C.clip_len:
                ret, f = cap.read()
                if not ret: break
                fs.append(cv2.resize(cv2.cvtColor(f, cv2.COLOR_BGR2RGB),
                                     (C.face_W, C.face_H)))
            cap.release()
            if not fs:
                arr = np.random.randint(100, 200, (C.clip_len, C.face_H, C.face_W, 3), np.uint8)
            else:
                while len(fs) < C.clip_len: fs.append(fs[-1].copy())
                arr = np.stack(fs[:C.clip_len])
        else:
            arr = np.random.randint(100, 200, (C.clip_len, C.face_H, C.face_W, 3), np.uint8)

        # Resize to model input if needed
        if arr.shape[1] != C.face_H or arr.shape[2] != C.face_W:
            arr = np.stack([cv2.resize(f, (C.face_W, C.face_H)) for f in arr])

        clip = torch.from_numpy(arr.astype(np.float32) / 255.0).permute(0, 3, 1, 2)
        if clip.shape[0] < C.clip_len:
            pad  = torch.zeros(C.clip_len - clip.shape[0], 3, C.face_H, C.face_W)
            clip = torch.cat([clip, pad], 0)
        return clip[:C.clip_len]


n_tr = int(0.85 * len(recs))
train_ds = rPPGDataset(recs[:n_tr], is_train=True)
val_ds   = rPPGDataset(recs[n_tr:], is_train=False)

_dl_kw = dict(
    batch_size=C.batch_size,
    num_workers=C.num_workers,
    pin_memory=True,
    persistent_workers=(C.num_workers > 0),
    prefetch_factor=2 if C.num_workers > 0 else None,
)
train_dl = DataLoader(train_ds, shuffle=True,  **_dl_kw)
val_dl   = DataLoader(val_ds,   shuffle=False, **_dl_kw)

console.print(f'[bold]Train: {len(train_ds)} | Val: {len(val_ds)}[/]')

In [ ]:
# ── PhysNet + Squeeze-Excitation (v2)

class SqueezeExcitation3D(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool3d(1), nn.Flatten(),
            nn.Linear(channels, max(1, channels//reduction)), nn.ReLU(inplace=True),
            nn.Linear(max(1, channels//reduction), channels), nn.Sigmoid(),
        )
    def forward(self, x):
        B, C_, T, H, W = x.shape
        return x * self.se(x).view(B, C_, 1, 1, 1)

class ConvSE3D(nn.Module):
    def __init__(self, in_c, out_c, k=(1,3,3), s=1, p=(0,1,1), se_red=16):
        super().__init__()
        self.conv = nn.Conv3d(in_c, out_c, k, stride=s, padding=p, bias=False)
        self.bn   = nn.BatchNorm3d(out_c)
        self.act  = nn.GELU()
        self.se   = SqueezeExcitation3D(out_c, se_red)
    def forward(self, x):
        return self.se(self.act(self.bn(self.conv(x))))

class PhysNetSE(nn.Module):
    def __init__(self, embed_dim=256, se_red=16, dp=0.3):
        super().__init__()
        SR = se_red
        self.blocks = nn.Sequential(
            ConvSE3D(3,   16,  (1,5,5), p=(0,2,2), se_red=SR),
            nn.MaxPool3d((1,2,2),(1,2,2)),
            ConvSE3D(16,  32,  (3,3,3), p=(1,1,1), se_red=SR),
            nn.MaxPool3d((1,2,2),(1,2,2)),
            ConvSE3D(32,  64,  (3,3,3), p=(1,1,1), se_red=SR),
            nn.MaxPool3d((1,2,2),(1,2,2)),
            ConvSE3D(64, 128,  (3,3,3), p=(1,1,1), se_red=SR),
            nn.AdaptiveAvgPool3d((None,4,4)),
        )
        self.temporal = nn.Sequential(
            nn.Conv1d(128*16, 256, 5, padding=2), nn.GELU(),
            nn.Conv1d(256, 128, 3, padding=1),    nn.GELU(),
            nn.Dropout(dp),
        )
        self.bvp_head   = nn.Conv1d(128, 1, 1)
        self.embed_head = nn.Sequential(
            nn.AdaptiveAvgPool1d(64), nn.Flatten(),
            nn.Linear(128*64, 512), nn.GELU(), nn.Dropout(dp),
            nn.Linear(512, embed_dim), nn.LayerNorm(embed_dim),
        )
        self.hr_head  = nn.Sequential(nn.Linear(embed_dim,64), nn.GELU(), nn.Linear(64,2))
        self.bp_head  = nn.Sequential(nn.Linear(embed_dim,64), nn.GELU(), nn.Linear(64,2))
        self.hrv_head = nn.Sequential(nn.Linear(embed_dim,64), nn.GELU(), nn.Linear(64,2))
        self.log_vars = nn.Parameter(torch.zeros(3))

    def _blocks_forward(self, x):
        return self.blocks(x)

    def forward(self, clip):
        B, T, C_, H, W = clip.shape
        x = clip.permute(0,2,1,3,4)  # (B,3,T,H,W)
        if self.training:
            import torch.utils.checkpoint as gc
            x = gc.checkpoint(self._blocks_forward, x, use_reentrant=False)
        else:
            x = self._blocks_forward(x)
        B2,C2,T2,H2,W2 = x.shape
        x     = x.view(B2, C2*H2*W2, T2)
        x     = self.temporal(x)
        bvp   = self.bvp_head(x).squeeze(1)
        embed = self.embed_head(x)
        return {'bvp': bvp, 'embed': embed,
                'hr':  self.hr_head(embed),
                'bp_systolic': self.bp_head(embed),
                'hrv_sdnn':    self.hrv_head(embed),
                'log_vars':    self.log_vars}


# ── FIX: compile BEFORE DataParallel
model = PhysNetSE(embed_dim=C.embed_dim, se_red=C.se_reduction, dp=C.dropout).to(DEVICE)

try:
    model = torch.compile(model, mode='reduce-overhead')
    console.print('[green]✅ torch.compile[/]')
except Exception as e:
    console.print(f'[yellow]torch.compile skipped: {e}[/]')

if NUM_GPUS > 1:
    model = nn.DataParallel(model)
    console.print(f'[green]✅ DataParallel across {NUM_GPUS} GPUs[/]')

console.print(f'Params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M')

with torch.no_grad():
    _d = torch.randn(2, C.clip_len, 3, C.face_H, C.face_W).to(DEVICE)
    _o = model(_d)
    console.print(f"BVP: {_o['bvp'].shape} | Embed: {_o['embed'].shape}")
del _d, _o; torch.cuda.empty_cache()

In [ ]:
import json
# ── Loss functions
def neg_pearson(pred, target):
    pm = pred   - pred.mean(-1, keepdim=True)
    tm = target - target.mean(-1, keepdim=True)
    corr = (pm*tm).sum(-1) / (pm.pow(2).sum(-1).sqrt() * tm.pow(2).sum(-1).sqrt() + 1e-8)
    return (1-corr).mean()

def gaussian_nll(pred_out, target):
    mu, lv = pred_out[:,0], pred_out[:,1]
    var = lv.exp().clamp(1e-6)
    return (0.5*(lv + (mu-target)**2/var)).mean()

def kg_weight(loss, log_var):
    return (torch.exp(-log_var)*loss + log_var).mean()

def compute_loss(out, batch):
    bvp_loss = neg_pearson(out['bvp'], batch['bvp'].to(DEVICE))
    hr_loss  = gaussian_nll(out['hr'].float(),          batch['hr'].to(DEVICE))
    bp_loss  = gaussian_nll(out['bp_systolic'].float(), batch['bp_systolic'].to(DEVICE))
    hrv_loss = gaussian_nll(out['hrv_sdnn'].float(),    batch['hrv_sdnn'].to(DEVICE))
    lv = out['log_vars']
    if lv.dim() > 1: lv = lv.mean(0)   # DataParallel stacks per-GPU
    return (0.5*bvp_loss
            + kg_weight(hr_loss,  lv[0])
            + kg_weight(bp_loss,  lv[1])
            + kg_weight(hrv_loss, lv[2]))


class EMA:
    def __init__(self, m, d=0.9995):
        self.d = d
        raw = self._unwrap(m)
        self.shadow = {n: p.data.clone().cpu()
                       for n, p in raw.named_parameters() if p.requires_grad}

    @staticmethod
    def _unwrap(m):
        raw = m.module if hasattr(m, 'module') else m
        try: raw = raw._orig_mod
        except AttributeError: pass
        return raw

    def update(self, m):
        raw = self._unwrap(m)
        for n, p in raw.named_parameters():
            if p.requires_grad and n in self.shadow:
                self.shadow[n] = self.d*self.shadow[n] + (1-self.d)*p.data.cpu()


try:
    optimiser = bnb.optim.AdamW8bit(model.parameters(), lr=C.lr, weight_decay=C.weight_decay)
    console.print('[green]✅ AdamW-8bit[/]')
except Exception:
    optimiser = torch.optim.AdamW(model.parameters(), lr=C.lr, weight_decay=C.weight_decay)
    console.print('[yellow]Fallback: AdamW FP32[/]')

scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=C.epochs, eta_min=1e-6)
scaler_amp = torch.amp.GradScaler('cuda')
ema        = EMA(model, C.ema_decay)

start_ep = 1; best_val = float('inf'); history = []
if RESUME_CKPT:
    try:
        ckpt = torch.load(RESUME_CKPT, map_location=DEVICE)
        EMA._unwrap(model).load_state_dict(ckpt['model_state'])
        optimiser.load_state_dict(ckpt['opt_state'])
        scheduler.load_state_dict(ckpt['sch_state'])
        scaler_amp.load_state_dict(ckpt['scaler_state'])
        ema.shadow = ckpt['ema_shadow']; history = ckpt['history']
        best_val = ckpt['best_val']; start_ep = ckpt['epoch'] + 1
        console.print(f'[green]✅ Resumed from epoch {ckpt["epoch"]}[/]')
    except Exception as e:
        console.print(f'[red]Resume failed: {e}[/]')

patience_cnt = 0
console.print(f'[bold]Training {C.epochs} epochs | batch={C.batch_size} | accum={C.grad_accum}[/]')


def run_ep(loader, train=True):
    model.train(train); tl, errs = 0.0, []
    if train: optimiser.zero_grad(set_to_none=True)
    for step, batch in enumerate(loader):
        clip = batch['clip'].to(DEVICE, non_blocking=True)
        with torch.autocast('cuda', dtype=DTYPE):
            out  = model(clip)
            loss = compute_loss(out, batch) / C.grad_accum
        if train:
            scaler_amp.scale(loss).backward()
            if (step+1) % C.grad_accum == 0:
                scaler_amp.unscale_(optimiser)
                nn.utils.clip_grad_norm_(model.parameters(), C.grad_clip)
                scaler_amp.step(optimiser); scaler_amp.update()
                optimiser.zero_grad(set_to_none=True); ema.update(model)
        tl += loss.item() * C.grad_accum
        for bi in range(out['bvp'].shape[0]):
            bvp_np = out['bvp'][bi].detach().float().cpu().numpy()
            errs.append(abs(bvp_to_hr(bvp_np) - batch['hr'][bi].item()))
    return tl/len(loader), float(np.mean(errs))


for epoch in range(start_ep, C.epochs+1):
    t0 = time.time()
    tr_l, tr_e = run_ep(train_dl, train=True)
    with torch.no_grad(): vl_l, vl_e = run_ep(val_dl, train=False)
    scheduler.step()
    history.append({'epoch':epoch,'train_loss':tr_l,'val_loss':vl_l,'val_hr_mae':vl_e})
    console.print(f'Ep {epoch:02d}/{C.epochs} | loss={vl_l:.4f} | HR_MAE={vl_e:.1f}bpm | {time.time()-t0:.0f}s')

    raw_m = EMA._unwrap(model)
    save_checkpoint(epoch, raw_m.state_dict(), ema.shadow, optimiser.state_dict(),
                    scheduler.state_dict(), scaler_amp.state_dict(), history, vl_l)
    if vl_l < best_val:
        best_val, patience_cnt = vl_l, 0
        torch.save({'model_state': raw_m.state_dict(), 'ema_shadow': ema.shadow, 'val_hr_mae': vl_e},
                   CKPT_DIR/'rppg_best.pt')
        console.print('  [green]✅ Best saved[/]')
    else:
        patience_cnt += 1
        if patience_cnt >= C.patience:
            console.print('[yellow]Early stop[/]'); break

json.dump(history, open(Path(C.out_dir)/'rppg_history.json','w'))
console.print('[bold green]\n✅ rPPG training done![/]')

In [ ]:
# ── Load best, apply EMA, export FP32 → INT8 ONNX
ckpt  = torch.load(CKPT_DIR/'rppg_best.pt', map_location='cpu')
raw_m = EMA._unwrap(model)
raw_m.load_state_dict(ckpt['model_state'])
for n, p in raw_m.named_parameters():
    if p.requires_grad and n in ckpt['ema_shadow']:
        p.data.copy_(ckpt['ema_shadow'][n])
raw_m = raw_m.float().cpu().eval()

class rPPGExport(nn.Module):
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, clip):
        o = self.m(clip)
        return o['embed'], o['bvp'], o['hr'][:,0], o['bp_systolic'][:,0], o['hrv_sdnn'][:,0]

exp_m = rPPGExport(raw_m).eval()
dummy = torch.randn(1, C.clip_len, 3, C.face_H, C.face_W)

fp32 = Path(C.out_dir)/'rppg_fp32.onnx'
torch.onnx.export(
    exp_m, dummy, fp32,
    input_names  =['video_clip'],
    output_names =['rppg_embedding','bvp_signal','hr_bpm','bp_systolic','hrv_sdnn'],
    dynamic_axes ={'video_clip':{0:'B'},'rppg_embedding':{0:'B'},'bvp_signal':{0:'B'},
                   'hr_bpm':{0:'B'},'bp_systolic':{0:'B'},'hrv_sdnn':{0:'B'}},
    opset_version=17, do_constant_folding=True,
)
onnx.checker.check_model(onnx.load(str(fp32)))

int8 = Path(C.out_dir)/'rppg_int8.onnx'
quantize_dynamic(str(fp32), str(int8), weight_type=QuantType.QInt8, optimize_model=True)
console.print(f'[bold green]✅ FP32={fp32.stat().st_size/1e6:.1f}MB → INT8={int8.stat().st_size/1e6:.1f}MB[/]')

# ── Training curves
h = pd.DataFrame(history)
fig, axes = plt.subplots(1, 2, figsize=(12,4))
axes[0].plot(h['epoch'], h['train_loss'], label='Train')
axes[0].plot(h['epoch'], h['val_loss'],   label='Val')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(h['epoch'], h['val_hr_mae'])
axes[1].axhline(5, color='green', ls='--', label='±5 bpm target')
axes[1].set_title('HR MAE (bpm)'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(Path(C.out_dir)/'rppg_training.png', dpi=150); plt.show()
console.print(f'[bold]rPPG embed dim: {C.embed_dim} → feeds fusion notebook[/]')